# Data Preprocessing

In [1]:
from pyspark.sql import SparkSession
import os

spark = SparkSession.builder \
    .appName("Final_Project_Data_Preprocessing") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/apps/software/standard/core/pyspark/4.2.0-py3.14/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/23 18:44:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/23 18:44:06 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
DATA_FILEPATH = '/standard/ds7200-apt4c/team2/ptb_superclass_v4'

In [3]:
df = spark.read.parquet(DATA_FILEPATH)
df.select("ecg_id").show(5)

+------+
|ecg_id|
+------+
|   217|
|   290|
|   441|
|   733|
|   800|
+------+
only showing top 5 rows


### Data Types

In [4]:
from collections import Counter

dtype_counts = Counter(dtype for col, dtype in df.dtypes)

print(dtype_counts)

Counter({'double': 1837, 'int': 228, 'string': 5})


### Duplicated IDs: ECGs and Patients

In [5]:
import pyspark.sql.functions as fn

df.agg(
    fn.count('ecg_id').alias('count ecgs'),
    fn.countDistinct('ecg_id').alias('distinct ecgs')
).show()

+----------+-------------+
|count ecgs|distinct ecgs|
+----------+-------------+
|     21378|        21378|
+----------+-------------+



In [12]:
df.agg(
    fn.count('patient_id').alias('count patients'),
    fn.countDistinct('patient_id').alias('distinct patients')
).show()

+--------------+-----------------+
|count patients|distinct patients|
+--------------+-----------------+
|         21378|            18610|
+--------------+-----------------+



### Missing Values

In [6]:
missing = df.agg(*[
    fn.sum(fn.col(c).isNull().cast('int')).alias(c)
    for c in df.columns
])

missing_count = (missing.select(
    fn.expr("stack({}, {})".format(
        len(df.columns),
        ", ".join([f"'{c}', `{c}`" for c in df.columns])))
    .alias("column", "missing_count"))
    .filter(fn.col("missing_count") > 100)
    .orderBy("missing_count", ascending=False)
)
missing_count.show(100)

26/11/23 18:44:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/11/23 18:44:38 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
[Stage 8:>                                                          (0 + 2) / 2]

ConnectionRefusedError: [Errno 111] Connection refused

In [14]:
missing_count.select("column").count()

26/11/21 15:25:20 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/11/21 15:25:35 WARN DAGScheduler: Broadcasting large task binary with size 4.9 MiB
                                                                                

100

In [37]:
from pyspark.sql import functions as F

col = "pacemaker"

df.select(col).groupby(col).agg(F.count(col)).orderBy(col).show(100)

+-------------+----------------+
|    pacemaker|count(pacemaker)|
+-------------+----------------+
|         NULL|               0|
|       ?, nan|               1|
|PACE????, nan|               2|
|      ja, nan|               3|
|ja, pacemaker|               7|
+-------------+----------------+



#### Imputing missing values and creating binary variables where applicable

For the infarction stadium fields, th data dictionary says: "extracted if present -- treat nulls as NA, not missing data"

In [15]:
df = df.fillna("NA", subset=["infarction_stadium1", "infarction_stadium2"])

For heart_axis, NLM study lists the missing values as unknown ("UNK")

In [30]:
df = df.fillna("UNK", subset=["heart_axis"])

Baseline drift, static noise, burst noise, extra_beats, electrodes_problems: Making these fields binary, where nulls are 0. Each is a string with a lot of categories. 

In [ ]:
from pyspark.sql.functions import col, when

string_cols = ["electrodes_problems", "burst_noise", "baseline_drift", "extra_beats", "static_noise"]

for c in string_cols:
    df = df.withColumn(c, when(col(c).isNotNull(), 1).otherwise(0))

#### Removing columns with thousands of missing values?

In [38]:
high_missing = missing_count.filter(missing_count.missing_count >= 3000)
remove_cols = [row["column"] for row in high_missing.select("column").collect()]

# df = df.select([
#     c for c in df.columns if ~(c.isin(remove_cols))
# ])

26/11/21 16:24:03 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/11/21 16:24:17 WARN DAGScheduler: Broadcasting large task binary with size 4.9 MiB
                                                                                

### Outliers?

In [25]:
df_outliers = df.select('ecg_id', 'age', 'height', 'weight').dropna()

cols = ['age', 'height', 'weight']
bounds = {}

for col in cols:
    quantiles = df_outliers.approxQuantile(
        col, [0.25, 0.75], 0.05
    )
    
    IQR = quantiles[1] - quantiles[0]
    
    bounds[col] = [
        quantiles[0] - 1.5 * IQR, 
        quantiles[1] + 1.5 * IQR
]

outliers = df_outliers.select(*['ecg_id'] + [
    (
        (df[c] < bounds[c][0]) | 
        (df[c] > bounds[c][1])
    ).alias(c + '_o') for c in cols
])
outliers.show()

+------+-----+--------+--------+
|ecg_id|age_o|height_o|weight_o|
+------+-----+--------+--------+
|   116|false|   false|   false|
|   136|false|   false|   false|
|   156|false|   false|   false|
|   256| true|   false|   false|
|   260|false|   false|   false|
|   261|false|   false|   false|
|   264|false|   false|   false|
|   265|false|   false|   false|
|   267|false|   false|   false|
|   270|false|   false|   false|
|   272|false|   false|   false|
|   273|false|   false|   false|
|   275|false|   false|   false|
|   276|false|   false|   false|
|   277|false|   false|   false|
|   278|false|   false|   false|
|   281|false|   false|   false|
|   284|false|   false|   false|
|   287|false|   false|   false|
|   288|false|   false|   false|
+------+-----+--------+--------+
only showing top 20 rows


In [26]:
df_outliers = df_outliers.join(outliers, on='ecg_id')
df_outliers.filter('weight_o').select('ecg_id', 'weight').show()
df_outliers.filter('age_o').select('ecg_id', 'age').show()

+------+------+
|ecg_id|weight|
+------+------+
|   751| 130.0|
|   856| 110.0|
|   911| 110.0|
|   975| 109.0|
|  1000| 120.0|
|  1083| 108.0|
|  1150| 118.0|
|  1185| 118.0|
|  1195| 112.0|
|  1213| 106.0|
|  1393| 110.0|
|  1625|  32.0|
|  1983| 119.0|
|  2081| 110.0|
|  2976| 140.0|
|  3526| 124.0|
|  3961|  25.0|
|  4634| 108.0|
|  4691| 109.0|
|  4759| 116.0|
+------+------+
only showing top 20 rows
+------+-----+
|ecg_id|  age|
+------+-----+
|   256|300.0|
|   351|300.0|
|   420| 22.0|
|   569|300.0|
|   637|300.0|
|   645| 19.0|
|   712|300.0|
|   730|300.0|
|   873|300.0|
|   932|300.0|
|  1075|300.0|
|  1184| 21.0|
|  1236| 20.0|
|  1502|300.0|
|  1625|300.0|
|  1657| 12.0|
|  1782| 22.0|
|  1792|300.0|
|  1970| 17.0|
|  1992|300.0|
+------+-----+
only showing top 20 rows


According to the data dictionary, "values above 89 will be in the reange of 300, for privacy -- they are not outliers". We will encode these as ... 

(should we use 90?)

In [ ]:
#df = df.filter('age <= 120')

### String Features

In [44]:
string_cols = ['report', 'scp_codes', 'heart_axis', 'infarction_stadium1', 'infarction_stadium2', 'second_opinion',
               'initial_autogenerated_report', 'validated_by_human', 'baseline_drift', 'static_noise', 'burst_noise',
               'electrodes_problems', 'extra_beats', 'pacemaker', 'strat_fold', 'filename_lr', 'filename_hr']

In [50]:
from pyspark.sql import functions as F

col = "scp_codes"

df.select(col).groupby(col).agg(F.count(col)).orderBy(col).show(100)

+--------------------+----------------+
|           scp_codes|count(scp_codes)|
+--------------------+----------------+
|{'1AVB': 100.0, '...|               1|
|{'1AVB': 100.0, '...|               1|
|{'1AVB': 100.0, '...|               1|
|{'1AVB': 100.0, '...|               1|
|{'1AVB': 100.0, '...|               1|
|{'1AVB': 100.0, '...|               1|
|{'1AVB': 100.0, '...|               2|
|{'1AVB': 100.0, '...|               1|
|{'1AVB': 100.0, '...|               1|
|{'1AVB': 100.0, '...|               1|
|{'1AVB': 100.0, '...|               1|
|{'1AVB': 100.0, '...|               1|
|{'1AVB': 100.0, '...|               1|
|{'1AVB': 100.0, '...|               1|
|{'1AVB': 100.0, '...|               1|
|{'1AVB': 100.0, '...|               1|
|{'1AVB': 100.0, '...|               1|
|{'1AVB': 100.0, '...|               1|
|{'1AVB': 100.0, '...|               4|
|{'1AVB': 100.0, '...|               1|
|{'1AVB': 100.0, '...|               1|
|{'1AVB': 100.0, '...|               1|


Notes on values:

READY FOR FINAL DATASET:
 - heart_axis - 9 acronyms, 8k unknown
 - infarction_stadium1 - 6 values, confirmed that nulls are NA/imputed in previous step
 - infarction_stadium2 - 3 values, confirmed that nulls are NA/imputed in previous step
 - baseline_drift - converted to binary with missing = 0 and filled = 1
 - static_noise - same as ^
 - burst_noise - same as ^
 - extra_beats - same as ^
 - electrodes_problems - same as ^
 - pacemaker - same as ^
 - second_opinion - removed, 21k False
 - initial_autogenerated_report - convert to binary, records either true or false
 - validated_by_human - same as ^
 - filename_lr AND filename_hr - drop, this is basically an id

#### Convert string columns to binary where the values are either true or false

In [ ]:
string_cols2 = ["second_opinion", "validated_by_human", "initial_autogenerated_report"]

for c in string_cols2:
    df = df.withColumn(c, (col(c) == "true").cast("integer"))